Realización del EDA en referente al dataset bank-full.csv dataset:

Explicación de las variables de dicho dataset:

## 📊 Diccionario de Datos (Data Dictionary)

| Variable | Rol | Tipo de Dato | Tipo Democrático / Descripción | Unidades | ¿Valores Faltantes? |
| :--- | :--- | :--- | :--- | :--- | :---: |
| **`age`** | Feature | Integer | Edad del cliente | Años | No |
| **`job`** | Feature | Categorical | Tipo de trabajo (`admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown`) | — | No |
| **`marital`** | Feature | Categorical | Estado civil (`divorced`, `married`, `single`, `unknown`; nota: *divorced* incluye viudos) | — | No |
| **`education`** | Feature | Categorical | Nivel educativo (`basic.4y`, `basic.6y`, `basic.9y`, `high.school`, `illiterate`, `professional.course`, `university.degree`, `unknown`) | — | No |
| **`default`** | Feature | Binary | ¿Tiene crédito en mora/incumplimiento? (`yes`, `no`, `unknown`) | — | No |
| **`balance`** | Feature | Integer | Saldo medio anual | Euros (€) | No |
| **`housing`** | Feature | Binary | ¿Tiene préstamo hipotecario? (`yes`, `no`, `unknown`) | — | No |
| **`loan`** | Feature | Binary | ¿Tiene préstamo personal? (`yes`, `no`, `unknown`) | — | No |
| **`contact`** | Feature | Categorical | Tipo de comunicación de contacto (`cellular`, `telephone`) | — | **Sí** |
| **`day_of_week`** | Feature | Date / Categorical | Último día de la semana en que fue contactado | Días | No |
| **`month`** | Feature | Date / Categorical | Último mes del año en que fue contactado (`jan`, `feb`, ..., `nov`, `dec`) | Meses | No |
| **`duration`** | Feature | Integer | Duración del último contacto. | Segundos | No |
| **`campaign`** | Feature | Integer | Número de contactos realizados durante esta campaña para este cliente (incluye el último) | Contactos | No |
| **`pdays`** | Feature | Integer | Días transcurridos desde que el cliente fue contactado en una campaña previa (`-1` o `999` indica no contactado previamente) | Días | **Sí** |
| **`previous`** | Feature | Integer | Número de contactos realizados antes de esta campaña para este cliente | Contactos | No |
| **`poutcome`** | Feature | Categorical | Resultado de la campaña de marketing anterior (`failure`, `nonexistent`, `success`) | — | **Sí** |
| **`y`** | **Target** | Binary | **Variable objetivo:** ¿El cliente suscribió un depósito a plazo fijo? (`yes`, `no`) | — | No |

In [ ]:
import pandas as pd  #for data manipulation operations
import numpy as np  #for numeric operations on data
import seaborn as sns  #for data visualization operations
import matplotlib.pyplot as plt  #for data visualization operations
from sklearn.preprocessing import LabelEncoder # for encoding
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler #for standardization
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import scipy.stats as st
from termcolor import colored

#from markupsafe import escape
#!pip install pandas-profiling
#import pandas_profiling

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import plot_confusion_matrix
from sklearn import model_selection
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
#!pip install lightgbm
from lightgbm import LGBMClassifier

#ignore warnings
import warnings
warnings.filterwarnings("ignore")

#see model parametres
from sklearn import set_config
set_config(print_changed_only = False)

print(colored("\n THE REQUIRED LIBRARIES WERE SUCCESFULLY IMPORTED...", "green"))

In [4]:
# Cargamos el dataset

data = pd.read_csv('data/bank-full.csv', sep=';')

data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
data['balance'].max()

np.int64(102127)

In [ ]:
# Exploramos si existen nulos

data.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [16]:
unknown_counts = (data == 'unknown').sum()
unknown_percentage = (data == 'unknown').mean() * 100

In [17]:
unknown_counts

age              0
job            288
marital          0
education     1857
default          0
balance          0
housing          0
loan             0
contact      13020
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome     36959
y                0
dtype: int64

In [18]:
unknown_percentage

age           0.000000
job           0.637013
marital       0.000000
education     4.107407
default       0.000000
balance       0.000000
housing       0.000000
loan          0.000000
contact      28.798301
day           0.000000
month         0.000000
duration      0.000000
campaign      0.000000
pdays         0.000000
previous      0.000000
poutcome     81.747805
y             0.000000
dtype: float64

El dataset no contiene valores missing explícitos (NaN), pero algunas variables contienen valores especiales que representan ausencia o desconocimiento de información, como pdays con -1 (que signicia que no se contactó, información util) y poutcome con "unknown". Este última podríamos tratarlo como NaN, pero en prinicipio para realizar el EDA lo mantendremos como una categoría más con el fin de ver si tiene alguna relación sustancial conocida con la variable y que nuestro modelo le venga bien entender.

In [7]:
# Comprobamos que no existan duplicados

data.duplicated().sum()

np.int64(0)

In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [ ]:
# Revisamos los types de cada variable para luego preparar los datos

categorical_columns = data.select_dtypes(include="str").columns

data[categorical_columns] = data[categorical_columns].astype("object")

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB
<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column 

In [13]:
data.nunique()

age            77
job            12
marital         3
education       4
default         2
balance      7168
housing         2
loan            2
contact         3
day            31
month          12
duration     1573
campaign       48
pdays         559
previous       41
poutcome        4
y               2
dtype: int64

| Variable       | Nº de categorías |
| -------------- | ---------------: |
| `job`          |               12 |
| `marital`      |                3 |
| `education`    |                4 |
| `default`      |                2 |
| `housing`      |                2 |
| `loan`         |                2 |
| `contact`      |                3 |
| `month`        |               12 |
| `poutcome`     |                4 |
| `y` *(target)* |                2 |


In [15]:
data.describe().T.style.background_gradient(cmap = "magma")

,count,mean,std,min,25%,50%,75%,max
age,45211.000000,40.936210,10.618762,18.000000,33.000000,39.000000,48.000000,95.000000
balance,45211.000000,1362.272058,3044.765829,-8019.000000,72.000000,448.000000,1428.000000,102127.000000
day,45211.000000,15.806419,8.322476,1.000000,8.000000,16.000000,21.000000,31.000000
duration,45211.000000,258.163080,257.527812,0.000000,103.000000,180.000000,319.000000,4918.000000
campaign,45211.000000,2.763841,3.098021,1.000000,1.000000,2.000000,3.000000,63.000000
pdays,45211.000000,40.197828,100.128746,-1.000000,-1.000000,-1.000000,-1.000000,871.000000
previous,45211.000000,0.580323,2.303441,0.000000,0.000000,0.000000,0.000000,275.000000
